# DARNet hyperparameter ablation plotting

This notebook parses the three DARNet hyperparameter/ablation log files and regenerates one PDF per experiment, dataset, prediction length, and metric.


In [1]:

from pathlib import Path
from collections import defaultdict, OrderedDict
import csv
import math
import re

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == 'Ablation' else Path.cwd()
OUT_DIR = ROOT / 'Ablation'
FIG_DIR = OUT_DIR / 'figures'
OUT_DIR.mkdir(exist_ok=True)
FIG_DIR.mkdir(exist_ok=True)

LOG_FILES = [
    ROOT / 'CPU-x86_64_GPU-NVIDIA-GeForce-RTX-4090_DARNet_hp_retrieval_topk.log',
    ROOT / 'CPU-x86_64_GPU-NVIDIA-GeForce-RTX-4090_DARNet_hp_num_experts.log',
    ROOT / 'CPU-x86_64_GPU-NVIDIA-GeForce-RTX-4090_DARNet_hp_state_prior_scales.log',
]
LOG_FILES = [p for p in LOG_FILES if p.exists()]

EXPERIMENT_META = {
    'DARNet_hp_retrieval_topk': {'name': 'retrieval_topk', 'label': 'Retrieval topK', 'x_config': 'Retrieval_Num', 'x_label': 'Retrieval topK'},
    'DARNet_hp_num_experts': {'name': 'num_experts', 'label': 'Number of Experts', 'x_config': 'Num_Experts', 'x_label': 'Number of experts'},
    'DARNet_hp_state_prior_scales': {'name': 'state_prior_scales', 'label': 'State Prior Scales', 'x_config': 'State_Prior_Setting', 'x_label': 'State prior scales'},
}

METRIC_ORDER = [
    'MAE', 'MSE', 'RMSE', 'MAPE', 'NMAE', 'NRMSE', 'COS','ECR_q90'
]

num_re = re.compile(r'^[-+]?\d+(?:\.\d+)?(?:[eE][-+]?\d+)?$')
metric_re = re.compile(r'([A-Za-z][A-Za-z0-9_]*)\s*-\s*([-+]?\d+(?:\.\d+)?(?:[eE][-+]?\d+)?)')

def strip_timestamp(line):
    return re.sub(r'^\|[^|]+\|\s*', '', line.strip())

def coerce_value(value):
    value = value.strip()
    if value == 'True':
        return True
    if value == 'False':
        return False
    if ',' in value:
        return value
    if num_re.match(value):
        return int(value) if re.match(r'^[-+]?\d+$', value) else float(value)
    return value

def parse_config(body):
    cfg = OrderedDict()
    for part in body.split(', '):
        if ' : ' not in part:
            continue
        key, value = part.split(' : ', 1)
        cfg[key.strip()] = coerce_value(value)
    return cfg

def parse_metrics(body):
    return OrderedDict((k, float(v)) for k, v in metric_re.findall(body))

def experiment_from_header(header_line, path):
    if header_line.startswith('#'):
        key = header_line.lstrip('#').strip()
        if key in EXPERIMENT_META:
            return key
    for key in EXPERIMENT_META:
        if key in path.stem:
            return key
    return path.stem

def scale_setting(row):
    scales = str(row.get('State_Prior_Scales', '')).strip()
    include_seq = row.get('State_Prior_Include_Seq_Level', True)
    return f'{scales}_no_seq' if include_seq is False or str(include_seq) == 'False' else scales

def x_order_key(label):
    label = str(label)
    if label.endswith('_no_seq'):
        base = label[:-7]
        return (1000 + len([x for x in base.split(',') if x]), base)
    if ',' in label:
        return (len([x for x in label.split(',') if x]), label)
    try:
        return (0, float(label))
    except ValueError:
        return (999, label)

def safe_name(value):
    return re.sub(r'[^A-Za-z0-9_.-]+', '_', str(value)).strip('_')


In [2]:

def parse_logs(log_files):
    rows = []
    for path in log_files:
        lines = [line for line in path.read_text(encoding='utf-8', errors='replace').splitlines() if line.strip()]
        header = next((line.strip() for line in lines if line.strip().startswith('#')), '')
        exp_key = experiment_from_header(header, path)
        meta = EXPERIMENT_META.get(exp_key, {'name': exp_key, 'label': exp_key, 'x_config': '', 'x_label': ''})
        pending_config = None
        for line in lines:
            body = strip_timestamp(line)
            if body.startswith('#'):
                continue
            if body.startswith('Dataset :'):
                pending_config = parse_config(body)
                continue
            metrics = parse_metrics(body)
            if metrics and pending_config is not None:
                row = OrderedDict()
                row['experiment_key'] = exp_key
                row['experiment'] = meta['name']
                row['experiment_label'] = meta['label']
                row['source_log'] = path.name
                row.update(pending_config)
                row.update(metrics)
                row['State_Prior_Setting'] = scale_setting(row)
                x_config = meta.get('x_config')
                row['variable_name'] = x_config
                row['variable_label'] = meta.get('x_label', x_config)
                row['variable_value'] = row.get(x_config, '')
                rows.append(row)
                pending_config = None
    return rows

rows = parse_logs(LOG_FILES)
print(f'Parsed rows: {len(rows)}')
print('Logs:', [p.name for p in LOG_FILES])

metric_cols = [m for m in METRIC_ORDER if any(m in row for row in rows)]
for row in rows:
    for m in metric_cols:
        row.setdefault(m, math.nan)

def write_csv(path, rows):
    keys = []
    for row in rows:
        for key in row.keys():
            if key not in keys:
                keys.append(key)
    with path.open('w', newline='', encoding='utf-8-sig') as f:
        writer = csv.DictWriter(f, fieldnames=keys)
        writer.writeheader()
        writer.writerows(rows)

write_csv(OUT_DIR / 'parsed_hyperparameter_ablation_results.csv', rows)

groups = defaultdict(list)
for row in rows:
    key = (row.get('experiment'), row.get('Dataset'), row.get('Pred_Len'), row.get('variable_name'), str(row.get('variable_value')))
    groups[key].append(row)

agg_rows = []
for _, items in groups.items():
    base = OrderedDict((k, v) for k, v in items[0].items() if k not in metric_cols)
    for m in metric_cols:
        vals = [float(item[m]) for item in items if item.get(m) is not None and not (isinstance(item.get(m), float) and math.isnan(item.get(m)))]
        base[m] = sum(vals) / len(vals) if vals else math.nan
    agg_rows.append(base)

write_csv(OUT_DIR / 'parsed_hyperparameter_ablation_results_aggregated.csv', agg_rows)
print(f'Aggregated rows: {len(agg_rows)}')


Parsed rows: 192
Logs: ['CPU-x86_64_GPU-NVIDIA-GeForce-RTX-4090_DARNet_hp_retrieval_topk.log', 'CPU-x86_64_GPU-NVIDIA-GeForce-RTX-4090_DARNet_hp_num_experts.log', 'CPU-x86_64_GPU-NVIDIA-GeForce-RTX-4090_DARNet_hp_state_prior_scales.log']
Aggregated rows: 192


In [3]:

plt.rcParams.update({
    'font.family': 'DejaVu Sans',
    'axes.unicode_minus': False,
    'pdf.fonttype': 42,
    'ps.fonttype': 42,
    'figure.dpi': 160,
    'savefig.transparent': True,
    'figure.facecolor': 'none',
    'axes.facecolor': 'none',
})

index_rows = []
plot_count = 0
by_combo = defaultdict(list)
for row in agg_rows:
    by_combo[(row['experiment'], row.get('Dataset'), row.get('Pred_Len'))].append(row)

for (experiment, dataset, pred_len), items in sorted(by_combo.items(), key=lambda x: (str(x[0][0]), str(x[0][1]), int(x[0][2]))):
    exp_label = items[0].get('experiment_label', experiment)
    variable_label = items[0].get('variable_label', items[0].get('variable_name', 'variable'))
    subdir = FIG_DIR / safe_name(experiment) / safe_name(dataset) / f'pred_len_{safe_name(pred_len)}'
    subdir.mkdir(parents=True, exist_ok=True)
    items_sorted = sorted(items, key=lambda r: x_order_key(str(r.get('variable_value'))))
    x_labels = [str(r.get('variable_value')) for r in items_sorted]
    x_pos = list(range(len(x_labels)))

    for metric in metric_cols:
        y = []
        for row in items_sorted:
            try:
                val = float(row.get(metric, math.nan))
            except (TypeError, ValueError):
                val = math.nan
            y.append(val)
        if not any(not math.isnan(v) for v in y):
            continue

        fig, ax = plt.subplots(figsize=(6.2, 4.0), facecolor='none')
        fig.patch.set_alpha(0.0)
        ax.patch.set_alpha(0.0)
        ax.plot(x_pos, y, marker='o', linewidth=1.8, markersize=5.5, color='#1f77b4')
        ax.set_title(f'{exp_label} | {dataset} | PredLen={pred_len} | {metric}', fontsize=10)
        ax.set_xlabel(variable_label)
        ax.set_ylabel(metric)
        ax.set_xticks(x_pos)
        rotate = any(len(x) > 6 for x in x_labels)
        ax.set_xticklabels(x_labels, rotation=25 if rotate else 0, ha='right' if rotate else 'center')
        ax.grid(True, linestyle='--', linewidth=0.6, alpha=0.45)
        ax.margins(x=0.08, y=0.12)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        fig.tight_layout()

        filename = f'{safe_name(experiment)}_{safe_name(dataset)}_pred{safe_name(pred_len)}_{safe_name(metric)}.pdf'
        out_path = subdir / filename
        fig.savefig(out_path, bbox_inches='tight', transparent=True, facecolor='none', edgecolor='none')
        plt.close(fig)
        plot_count += 1
        index_rows.append({'experiment': experiment, 'dataset': dataset, 'pred_len': pred_len, 'metric': metric, 'file': str(out_path.relative_to(OUT_DIR)).replace('\\', '/')})

with (OUT_DIR / 'figure_index.csv').open('w', newline='', encoding='utf-8-sig') as f:
    writer = csv.DictWriter(f, fieldnames=['experiment', 'dataset', 'pred_len', 'metric', 'file'])
    writer.writeheader()
    writer.writerows(index_rows)

print(f'Generated PDF figures: {plot_count}')
print(f'Figure root: {FIG_DIR}')


Generated PDF figures: 288
Figure root: d:\Extreme\Ablation\figures
